In [1]:
print("Saeed")

Saeed


In [3]:
import tensorflow_hub as hub 

d:\NLP\NLP_Practice\.venv\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [ ]:
import sys
print(sys.executable)

d:\NLP\NLP_Practice\.venv\Scripts\python.exe


In [ ]:
import setuptools
print(setuptools.__version__)

80.10.2


In [6]:
from transformers import BertForMaskedLM , BertTokenizer

In [7]:
import torch 

In [8]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForMaskedLM.from_pretrained("bert-base-uncased")

d:\NLP\NLP_Practice\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kasf traders\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\NLP\NLP_Practice\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-

In [9]:
model.eval()

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [ ]:
sentence = "The capital of France is [MASK]. "

In [11]:
inputs = tokenizer(sentence, return_tensors="pt")

In [14]:
inputs

{'input_ids': tensor([[ 101, 1996, 3007, 1997, 2605, 2003,  103, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [12]:
with torch.no_grad():
    outputs = model(**inputs)

In [15]:
outputs

MaskedLMOutput(loss=None, logits=tensor([[[ -6.4346,  -6.4063,  -6.4097,  ...,  -5.7691,  -5.6326,  -3.7883],
         [-14.0119, -14.7240, -14.2120,  ..., -11.6976, -10.7304, -12.7617],
         [ -9.6561, -10.3125,  -9.7459,  ...,  -8.7782,  -6.6036, -12.6596],
         ...,
         [ -3.7861,  -3.8572,  -3.5644,  ...,  -2.5593,  -3.1093,  -4.3820],
         [-11.6598, -11.4274, -11.9266,  ...,  -9.8772, -10.2103,  -4.7594],
         [-11.7267, -11.7509, -11.8040,  ..., -10.5943, -10.9407,  -7.5151]]]), hidden_states=None, attentions=None)

In [19]:
mask_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
predictions = outputs.logits[0, mask_index].softmax(-1)
top5 = torch.topk(predictions, 5, dim=1)
deocodedlist = []
print("Sentence:", sentence)
for score, idx in zip(top5.values[0], top5.indices[0]):
    word = tokenizer.decode([idx])
    deocodedlist.append(word)
    print(f"  {word:12s}  confidence: {score:.2%}")

Sentence: The capital of France is [MASK].
  paris         confidence: 41.68%
  lille         confidence: 7.14%
  lyon          confidence: 6.34%
  marseille     confidence: 4.44%
  tours         confidence: 3.03%


In [20]:
deocodedlist

['paris', 'lille', 'lyon', 'marseille', 'tours']

In [21]:
corrected = sentence.replace("[MASK]" , deocodedlist[0])
corrected

'The capital of France is paris.'

In [16]:
mask_index

tensor([6])

In [17]:
predictions

tensor([[4.1095e-08, 3.8276e-08, 5.1294e-08,  ..., 1.4016e-07, 8.0858e-08,
         2.2647e-08]])

In [18]:
top5

torch.return_types.topk(
values=tensor([[0.4168, 0.0714, 0.0634, 0.0444, 0.0303]]),
indices=tensor([[ 3000, 22479, 10241, 16766,  7562]]))